# Phase 10 — Second base model (Qwen2.5-7B-Instruct), baseline only
Minimum scope for Phase 10: does the ceiling-effect pattern (Tier 1/3 near-ceiling, Tier 2 open) replicate on a different model, or is it Llama-specific? No optimization here — just the untouched baseline on Tier 2 and Tier 3, same tasks, same grader.

Single session, no restart needed (baseline doesn't alternate DSPy/QLoRA).

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show 0MiB used before continuing.**

## 1. Clone repo

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from huggingface_hub import login
login()

## 2. Load Qwen2.5-7B-Instruct (not gated, no license acceptance needed, unlike Llama)

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.agent_harness import load_model, run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from grader import grade_task

model, tok = load_model("Qwen/Qwen2.5-7B-Instruct")
print("Qwen model loaded.")

## 3. Tier 2 baseline on Qwen (full task set, matching how the Llama Tier 2 baseline was measured)

In [ ]:
from tasks.tier2 import TIER2_TASKS

tier2_results = []
for task in TIER2_TASKS:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=2, tool_calls=tool_calls, final_text=final_text)
    tier2_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

rate = sum(r["grade"]["success"] for r in tier2_results) / len(tier2_results)
print(f"\nQwen Tier 2 baseline success rate: {rate:.1%}")
print("(Compare to Llama's Tier 2 baseline: 41.7%)")

with open("results/tier2_baseline_qwen_results.json", "w") as f:
    json.dump(tier2_results, f, indent=2)

## 4. Tier 3 baseline on Qwen

In [ ]:
from tasks.tier3 import TIER3_TASKS

tier3_results = []
for task in TIER3_TASKS:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=3, tool_calls=tool_calls, final_text=final_text)
    tier3_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

rate = sum(r["grade"]["success"] for r in tier3_results) / len(tier3_results)
print(f"\nQwen Tier 3 baseline success rate: {rate:.1%}")
print("(Compare to Llama's Tier 3 baseline: 90.0%)")

with open("results/tier3_baseline_qwen_results.json", "w") as f:
    json.dump(tier3_results, f, indent=2)

## 5. (Optional, if time/GPU quota allow) Tier 1 baseline too, for completeness
Not strictly needed for the ceiling-effect generalization check (Tier 1 was already near-ceiling on Llama and unlikely to differ), but cheap to add if you have quota left.

In [ ]:
from tasks.tier1 import TIER1_TASKS

tier1_results = []
for task in TIER1_TASKS:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool)
    grade = grade_task(task, tier=1, tool_calls=tool_calls, final_text=final_text)
    tier1_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

rate = sum(r["grade"]["success"] for r in tier1_results) / len(tier1_results)
print(f"\nQwen Tier 1 baseline success rate: {rate:.1%}")
print("(Compare to Llama's Tier 1 baseline: 93.8%)")

with open("results/tier1_baseline_qwen_results.json", "w") as f:
    json.dump(tier1_results, f, indent=2)

## 6. Download and push

In [ ]:
from google.colab import files
files.download("results/tier2_baseline_qwen_results.json")
files.download("results/tier3_baseline_qwen_results.json")
# files.download("results/tier1_baseline_qwen_results.json")  # uncomment if you ran Section 5

Move the downloaded files into `results/` on your laptop, then:
```bash
git add results/tier2_baseline_qwen_results.json results/tier3_baseline_qwen_results.json
git commit -m "Phase 10: Qwen2.5-7B-Instruct baseline, Tier 2 + Tier 3"
git push
```

## Stretch goal (separate future session): DSPy + QLoRA on Qwen
If the baseline pattern replicates and you want to go further, the same seed-enabled dspy_optimize_tierN.py / qlora_finetune_tierN.py scripts work on Qwen too — just pass `"Qwen/Qwen2.5-7B-Instruct"` wherever `load_model(...)` or `MODEL_ID` currently says Llama. Not required for Phase 10's minimum scope.